# Week 7: rPPG quality gate, pipeline validation and deployment checks

**Purpose.** Fix the pipeline problems found in the week 7 simulation, prove the pipeline works, and block deployment unless real skill is shown.

**What this fixes:**
- Bad rPPG sessions (16.1% off by more than 5 BPM): a quality gate that needs no contact sensor at inference.
- Ridge scoring below the constant on temperature: predictions are re-centred on the training median.
- Silent leakage: every split is checked for subject overlap.
- Dataset lock-in: any dataset can be used through one adapter.

**What this cannot fix:** MCD's labels. PSM-25 is a stable trait and temperature SD is 0.133 °C. If the real signal is too weak, the notebook reports that and marks the model NOT APPROVED.

**Scope:** works on extracted session features. A new dataset needs the same week 3 feature extraction first. Transformers are not included.

In [ ]:
# ---- CONFIG: edit this cell only ----
import os
ROOT = os.environ.get('MCD_ROOT', '/content/drive/MyDrive/MCD_rPPG')
OUT = os.environ.get('WEEK7_OUT', f'{ROOT}/week7_outputs')
SIM_DIR = 'SIMULATED_populated_data_NOT_REAL_MCD'   # planted-label tables go here, never with real results
BAD_HR_BPM = 5.0        # a session is 'bad' if |rPPG HR - reference HR| exceeds this
GATE_THRESHOLD = 0.20   # reject sessions with P(bad) at or above this
MIN_COVERAGE = 0.70     # the model must keep at least this share of sessions
RHOS = [0, .15, .3, .45, .6, .8]
N_SEEDS = 3
BOOT = 400
if ROOT.startswith('/content/drive'):
    from google.colab import drive
    drive.mount('/content/drive')
os.makedirs(OUT, exist_ok=True); os.makedirs(f'{ROOT}/{SIM_DIR}', exist_ok=True)

Mounted at /content/drive


## 1. Dataset adapter
Every dataset becomes one table with the same columns. To add a dataset, write a `DatasetSpec`; nothing else changes.

In [ ]:
import json, warnings, numpy as np, pandas as pd, joblib
from dataclasses import dataclass, field
from scipy.stats import rankdata, spearmanr
from sklearn.base import clone
from sklearn.ensemble import HistGradientBoostingClassifier as HC, HistGradientBoostingRegressor as HR
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import GroupKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
warnings.filterwarnings('ignore')

QUALITY = ['rppg_face_missing_frac','rppg_fps_measured','rppg_snr_face','rppg_snr_forehead','rppg_snr_left_cheek',
           'rppg_snr_right_cheek','rppg_snr','rppg_hr_win_sd','rppg_hr_consistency','rppg_n_beats','rppg_ibi_ok_frac',
           'rppg_beat_consistency','rppg_skin_lum','hr_beat_gap']
SIGNAL = ['rppg_snr','rppg_hr','rppg_hr_win_sd','rppg_hr_consistency','rppg_ln_sdnn','rppg_ln_rmssd','rppg_ln_lf',
          'rppg_ln_hf','rppg_lf_hf','rppg_beat_consistency','rppg_rise_frac','rppg_half_width']

@dataclass
class DatasetSpec:
    name: str
    subject: str
    session: str
    labels: dict                       # {'stress': 'column in file'}
    rename: dict = field(default_factory=dict)   # {'their column': 'our feature name'}
    reference_hr: str = None           # contact HR column, if any
    demo: list = field(default_factory=list)

def standardise(df, spec):
    d = df.rename(columns=spec.rename).copy()
    d['subject'] = d[spec.subject].astype(str); d['session'] = d[spec.session].astype(str)
    for k, c in spec.labels.items(): d['y_' + k] = pd.to_numeric(d[c], errors='coerce')
    d['reference_hr'] = d[spec.reference_hr] if spec.reference_hr else np.nan
    if 'rppg_hr_beats' in d: d['hr_beat_gap'] = (d.rppg_hr - d.rppg_hr_beats).abs()
    allf = list(dict.fromkeys(QUALITY + SIGNAL))
    need = sorted(set(allf) - set(d.columns))
    if need: raise ValueError(f'{spec.name}: missing features {need}. Run the same feature extraction first.')
    d[allf] = d[allf].replace([np.inf, -np.inf], np.nan)
    return d.reset_index(drop=True)

MCD = DatasetSpec('MCD-rPPG', 'patient_id', 'step', {'stress':'stress','temperature':'temperature'},
                  reference_hr='ppg_hr', demo=['age','sex_m','bmi'])
def load_mcd():
    db = pd.read_csv(f'{ROOT}/db.csv').drop_duplicates(['patient_id','step'])
    db = db[['patient_id','step','stress','temperature','age','sex','bmi']]
    db['sex_m'] = db.sex.astype(str).str.lower().str.startswith('m').astype(int)
    f = pd.read_csv(f'{ROOT}/week3_stress_rppg600/features/session_features.csv')
    return standardise(f.merge(db, on=['patient_id','step']).sort_values(['patient_id','step']), MCD)
D = load_mcd(); print(D.shape, D.subject.nunique(), 'subjects')

(1200, 78) 600 subjects


## 2. Label viability
Checks whether each label can be learned at all, before any model runs.

In [ ]:
def label_report(d, targets):
    rows = []
    for t in targets:
        y = d['y_' + t]; ch = y.groupby(d.subject).nunique().gt(1).mean()
        rows.append(dict(target=t, sd=y.std(), constant_mae=(y - y.median()).abs().mean(), subjects_changing=ch,
                         ref_hr_spearman=spearmanr(d.reference_hr, y, nan_policy='omit')[0],
                         ln_rmssd_spearman=spearmanr(d.rppg_ln_rmssd, y, nan_policy='omit')[0]))
    return pd.DataFrame(rows).round(3)
TARGETS = list(MCD.labels); LR = label_report(D, TARGETS); LR.to_csv(f'{OUT}/label_report.csv', index=False); LR

,target,sd,constant_mae,subjects_changing,ref_hr_spearman,ln_rmssd_spearman
0,stress,1.458,1.154,0.007,0.002,0.011
1,temperature,0.133,0.093,0.173,0.082,0.002


## 3. Core: gated, subject-grouped evaluation
The gate and model are fitted on training subjects only. Each fold is checked for subject overlap.

In [ ]:
GATE = lambda: make_pipeline(SimpleImputer(strategy='median'), HC(max_depth=3, max_iter=150, learning_rate=.05, random_state=0))
class MedianRidge:
    """Ridge whose predictions are re-centred on the training median, because MAE rewards the median."""
    def fit(self, X, y):
        self.m = make_pipeline(SimpleImputer(strategy='median'), StandardScaler(), Ridge(30)).fit(X, y)
        self.shift = np.median(y - self.m.predict(X)); return self
    def predict(self, X): return self.m.predict(X) + self.shift
MODELS = {'hgb': lambda: make_pipeline(SimpleImputer(strategy='median'),
              HR(loss='absolute_error', max_iter=80, max_depth=3, learning_rate=.05, min_samples_leaf=20, random_state=0)),
          'median_ridge': MedianRidge}

def fit_gate(d):
    r = d.reference_hr.notna() & d.rppg_hr.notna()
    bad = ((d.rppg_hr - d.reference_hr).abs() > BAD_HR_BPM)[r].astype(int).values
    return GATE().fit(d.loc[r, QUALITY], bad)
def keep(gate, d):
    return (gate.predict_proba(d[QUALITY])[:, 1] < GATE_THRESHOLD) & d.rppg_hr.notna().values

def run_cv(d, y, cols=SIGNAL, model='hgb', gated=True, seed=0):
    y = np.asarray(y, float); g = d.subject.values; n = len(d)
    p = np.full(n, np.nan); b = np.full(n, np.nan); kept = np.zeros(n, bool)
    for tr, te in GroupKFold(5, shuffle=True, random_state=seed).split(d, y, g):
        assert not set(g[tr]) & set(g[te]), 'subject leakage'
        dt, de = d.iloc[tr], d.iloc[te]
        if gated:
            gt = fit_gate(dt); kt, ke = keep(gt, dt), keep(gt, de)
        else: kt, ke = np.ones(len(tr), bool), np.ones(len(te), bool)
        m = MODELS[model]().fit(dt.loc[kt, cols].values, y[tr][kt])
        p[te[ke]] = m.predict(de.loc[ke, cols].values); b[te[ke]] = np.median(y[tr][kt]); kept[te[ke]] = True
    ep, eb = np.abs(y - p)[kept], np.abs(y - b)[kept]; gk = g[kept]
    a, c = pd.Series(ep).groupby(gk).sum().values, pd.Series(eb).groupby(gk).sum().values
    r = np.random.default_rng(seed); ix = r.integers(0, len(a), (BOOT, len(a)))
    B = 100 * (1 - a[ix].sum(1) / c[ix].sum(1))
    return dict(mae=ep.mean(), constant=eb.mean(), skill=100 * (1 - ep.sum() / eb.sum()), lo=np.percentile(B, 2.5),
                hi=np.percentile(B, 97.5), spearman=spearmanr(p[kept], y[kept])[0], coverage=kept.mean())
def shuffle_subjects(d, y, seed):
    y = pd.Series(np.asarray(y, float)); first = y.groupby(d.subject.values).transform('first')
    u = pd.Series(first.values).groupby(d.subject.values).first()
    perm = pd.Series(np.random.default_rng(seed).permutation(u.values), index=u.index)
    return (y - first + pd.Series(d.subject.values).map(perm)).values

## 4. Quality gate check
Out-of-fold on subjects the gate never saw. The reference HR is used to train the gate, never at inference.

In [ ]:
def gate_report(d):
    r = d[d.reference_hr.notna() & d.rppg_hr.notna()].reset_index(drop=True)
    err = (r.rppg_hr - r.reference_hr).abs().values; p = np.zeros(len(r))
    for tr, te in GroupKFold(5, shuffle=True, random_state=0).split(r, err, r.subject):
        p[te] = fit_gate(r.iloc[tr]).predict_proba(r.iloc[te][QUALITY])[:, 1]
    k = p < GATE_THRESHOLD
    return pd.Series(dict(auc=roc_auc_score(err > BAD_HR_BPM, p), coverage=k.mean(),
        before_median=np.median(err), before_mean=err.mean(), before_bad=(err > BAD_HR_BPM).mean(),
        after_median=np.median(err[k]), after_mean=err[k].mean(), after_bad=(err[k] > BAD_HR_BPM).mean())).round(3)
GR = gate_report(D); GR.to_csv(f'{OUT}/gate_report.csv'); GR

,0
auc,0.969
coverage,0.811
before_median,0.732
before_mean,5.529
before_bad,0.161
after_median,0.586
after_mean,1.017
after_bad,0.029


## 5. Controls
- **Positive:** predicting contact HR must work. If it fails, the pipeline is broken.
- **Negative:** shuffled labels must show no skill.

In [ ]:
rows = []
for gated in (False, True):
    rows.append(dict(control='positive: contact HR', gated=gated, **run_cv(D, D.reference_hr.fillna(D.reference_hr.median()), gated=gated)))
    for t in TARGETS:
        rows.append(dict(control=f'negative: shuffled {t}', gated=gated, **run_cv(D, shuffle_subjects(D, D['y_'+t], 1), gated=gated)))
CTRL = pd.DataFrame(rows).round(3); CTRL.to_csv(f'{OUT}/controls.csv', index=False)
POS_OK = bool(CTRL[(CTRL.control.str.startswith('pos')) & CTRL.gated].lo.iloc[0] > 50)
NEG_OK = bool((CTRL[CTRL.control.str.startswith('neg') & CTRL.gated].lo <= 0).all())
print('positive control pass:', POS_OK, '| negative control pass:', NEG_OK); CTRL

positive control pass: True | negative control pass: True


,control,gated,mae,constant,skill,lo,hi,spearman,coverage
0,positive: contact HR,False,2.526,10.408,75.726,73.385,78.068,0.955,1.00
1,negative: shuffled stress,False,1.190,1.158,-2.805,-3.685,-1.980,-0.108,1.00
2,negative: shuffled temperature,False,0.101,0.099,-1.573,-3.759,0.242,0.009,1.00
3,positive: contact HR,True,1.229,9.817,87.483,86.127,88.552,0.988,0.81
4,negative: shuffled stress,True,1.179,1.136,-3.721,-4.984,-2.531,-0.098,0.81
5,negative: shuffled temperature,True,0.102,0.099,-3.692,-6.464,-1.209,-0.036,0.81


## 6. Sensitivity with planted labels
Real features, **synthetic labels**. This shows how strong a signal must be before this pipeline can see it. Results are saved only to the simulated folder.

In [ ]:
z = lambda v: (v - np.nanmean(v)) / np.nanstd(v)
def composite(d):
    s = z(z(d.reference_hr) - z(d.ppg_ln_rmssd)); s = pd.Series(s).fillna(0)
    return z(s.groupby(d.subject.values).transform('mean').values)
def plant(d, t, rho, seed):
    r = np.random.default_rng(seed + int(rho * 100) * 100); S = composite(d); ids = d.subject.unique()
    e = pd.Series(d.subject.values).map(pd.Series(r.normal(0, 1, len(ids)), index=ids)).values
    lat = rho * S + np.sqrt(1 - rho ** 2) * e + .02 * r.normal(0, 1, len(d))
    real = np.sort(d['y_' + t].values.astype(float))
    return real[(rankdata(lat, method='ordinal') - 1).astype(int)]
rows = []
for t in TARGETS:
    for rho in RHOS:
        for s in range(N_SEEDS):
            for gated in (False, True):
                rows.append(dict(target=t, rho=rho, seed=s, gated=gated, **run_cv(D, plant(D, t, rho, s), gated=gated, seed=s)))
SENS = pd.DataFrame(rows); SENS.to_csv(f'{ROOT}/{SIM_DIR}/week7_sensitivity_SIMULATED.csv', index=False)
SENS['detected'] = SENS.lo > 0
display(SENS.groupby(['target','gated','rho']).agg(skill=('skill','mean'), detected=('detected','mean')).round(2).unstack('rho'))
OBS = {t: abs(spearmanr(composite(D), D['y_'+t])[0]) for t in TARGETS}
MIN_RHO = {t: SENS[(SENS.target==t)&SENS.gated].groupby('rho').detected.mean().loc[lambda s: s>=.8].index.min() for t in TARGETS}
print('observed real rho:', {k: round(v,3) for k,v in OBS.items()}, '| minimum detectable rho:', MIN_RHO)

skill                                 detected            \
rho                0.00  0.15  0.30  0.45   0.60   0.80     0.00 0.15 0.30   
target      gated                                                            
stress      False -2.40 -1.82 -1.41  3.00   8.78  20.13      0.0  0.0  0.0   
            True  -3.29 -2.81 -0.90  1.77  10.02  22.02      0.0  0.0  0.0   
temperature False -3.66 -2.32 -1.42  1.01   6.65  14.05      0.0  0.0  0.0   
            True  -3.17 -2.61 -1.63  1.68   8.22  16.36      0.0  0.0  0.0   

                                    
rho                0.45  0.60 0.80  
target      gated                   
stress      False  0.33  1.00  1.0  
            True   0.00  1.00  1.0  
temperature False  0.33  1.00  1.0  
            True   0.00  0.67  1.0

observed real rho: {'stress': np.float64(0.022), 'temperature': np.float64(0.068)} | minimum detectable rho: {'stress': np.float64(0.6), 'temperature': np.float64(0.8)}


## 7. Real evaluation and verdict
A model is approved only if every check passes. These thresholds were fixed before seeing the gated results.

In [ ]:
rows = []
for t in TARGETS:
    y = D['y_'+t].values
    for m in MODELS:
        rows.append(dict(target=t, model=m, inputs='rPPG', gated=False, **run_cv(D, y, model=m, gated=False)))
        rows.append(dict(target=t, model=m, inputs='rPPG', gated=True, **run_cv(D, y, model=m)))
        rows.append(dict(target=t, model=m, inputs='demographics', gated=True, **run_cv(D, y, cols=MCD.demo, model=m)))
RES = pd.DataFrame(rows).round(3); RES.to_csv(f'{OUT}/real_results.csv', index=False); display(RES)
def verdict(res, t):
    best = res[(res.target==t)&(res.inputs=='rPPG')&res.gated].sort_values('lo').iloc[-1]
    demo = res[(res.target==t)&(res.inputs=='demographics')].skill.max()
    checks = {'positive control': POS_OK, 'negative control': NEG_OK, 'CI above constant': bool(best.lo > 0),
              'beats demographics': bool(best.skill > demo), 'coverage': bool(best.coverage >= MIN_COVERAGE)}
    return dict(target=t, model=best.model, skill=best.skill, lo=best.lo, hi=best.hi, coverage=best.coverage, checks=checks,
                approved=all(checks.values()),
                reason='signal below detection limit' if OBS[t] < MIN_RHO[t] else 'see checks')
V = {t: verdict(RES, t) for t in TARGETS}
json.dump(V, open(f'{OUT}/verdict.json','w'), indent=1, default=float)
for t, v in V.items(): print(t, 'APPROVED' if v['approved'] else 'NOT APPROVED', '|', v['reason'], '|', v['checks'])

,target,model,inputs,gated,mae,constant,skill,lo,hi,spearman,coverage
0,stress,hgb,rPPG,False,1.204,1.154,-4.311,-5.930,-2.901,-0.086,1.00
1,stress,hgb,rPPG,True,1.157,1.136,-1.865,-3.129,-0.427,-0.030,0.81
2,stress,hgb,demographics,True,1.139,1.136,-0.335,-3.562,2.820,0.163,0.81
3,stress,median_ridge,rPPG,False,1.183,1.154,-2.517,-3.555,-1.561,0.002,1.00
4,stress,median_ridge,rPPG,True,1.165,1.136,-2.574,-3.935,-1.389,0.034,0.81
5,stress,median_ridge,demographics,True,1.113,1.136,1.997,-1.263,5.053,0.193,0.81
6,temperature,hgb,rPPG,False,0.098,0.093,-4.847,-7.643,-2.544,-0.046,1.00
7,temperature,hgb,rPPG,True,0.098,0.095,-3.073,-6.025,-0.207,-0.019,0.81
8,temperature,hgb,demographics,True,0.097,0.095,-2.014,-5.154,0.518,0.011,0.81
9,temperature,median_ridge,rPPG,False,0.097,0.093,-4.334,-6.247,-2.912,0.040,1.00


stress NOT APPROVED | signal below detection limit | {'positive control': True, 'negative control': True, 'CI above constant': False, 'beats demographics': False, 'coverage': True}
temperature NOT APPROVED | signal below detection limit | {'positive control': True, 'negative control': True, 'CI above constant': False, 'beats demographics': False, 'coverage': True}


## 8. Model bundle
Saves the gate, model and verdict together. `predict_bundle` refuses unapproved models and returns no value for rejected sessions.

In [ ]:
def make_bundle(d, t, y, model='hgb', approved=False, note=''):
    gate = fit_gate(d); k = keep(gate, d)
    m = MODELS[model]().fit(d.loc[k, SIGNAL].values, np.asarray(y, float)[k])
    shift = getattr(m, 'shift', 0.0); m = getattr(m, 'm', m)   # plain sklearn objects, so the bundle loads anywhere
    return dict(target=t, gate=gate, model=m, shift=float(shift), quality=QUALITY, signal=SIGNAL, threshold=GATE_THRESHOLD, approved=approved,
                baseline=float(np.median(np.asarray(y, float)[k])), note=note)
def predict_bundle(b, d, allow_unapproved=False):
    if not b['approved'] and not allow_unapproved:
        raise PermissionError(f"{b['target']} model is NOT APPROVED: {b['note']}")
    k = (b['gate'].predict_proba(d[b['quality']])[:, 1] < b['threshold']) & d.rppg_hr.notna().values
    out = np.full(len(d), np.nan); out[k] = b['model'].predict(d.loc[k, b['signal']].values) + b['shift']
    return pd.DataFrame(dict(subject=d.subject, session=d.session, prediction=out, accepted=k))
for t in TARGETS:
    v = V[t]; b = make_bundle(D, t, D['y_'+t].values, v['model'], v['approved'], v['reason'])
    joblib.dump(b, f'{OUT}/bundle_{t}.joblib')
    try: predict_bundle(b, D.head(5)); print(t, 'bundle usable')
    except PermissionError as e: print('blocked:', e)

blocked: stress model is NOT APPROVED: signal below detection limit
blocked: temperature model is NOT APPROVED: signal below detection limit


## 9. Portability check on a held-out dataset
This proves the code path, not health performance. 20% of subjects are held out and rewritten with different column names, as if they came from another dataset. The label is **synthetic** (ρ = 0.8).

In [ ]:
ids = D.subject.unique(); r = np.random.default_rng(7); ext_ids = set(r.choice(ids, len(ids)//5, replace=False))
yp = plant(D, 'stress', .8, 0); isx = D.subject.isin(ext_ids).values
assert not set(D.subject[isx]) & set(D.subject[~isx])
b = make_bundle(D[~isx].reset_index(drop=True), 'SIMULATED_stress', yp[~isx], approved=True, note='portability test only')
renames = {c: 'x_' + c for c in sorted(set(QUALITY + SIGNAL + ['rppg_hr_beats']) - {'hr_beat_gap'})}
ext = D[isx].drop(columns=['hr_beat_gap']).rename(columns=renames).assign(SIMULATED_label=yp[isx])
EXT_SPEC = DatasetSpec('held-out demo', 'patient_id', 'step', {'sim':'SIMULATED_label'}, rename={v:k for k,v in renames.items()})
E = standardise(ext, EXT_SPEC); P = predict_bundle(b, E)
a = P.accepted.values; yy = E.y_sim.values
print(f'accepted {a.mean():.1%} | MAE {np.abs(yy[a]-P.prediction[a]).mean():.3f} vs constant {np.abs(yy[a]-b["baseline"]).mean():.3f}')
missing = E.drop(columns=['rppg_snr'])
try: predict_bundle(b, missing)
except Exception as e: print('missing feature blocked:', type(e).__name__)

accepted 80.0% | MAE 0.869 vs constant 1.115
missing feature blocked: KeyError


## 10. Summary
To run a new dataset:
1. Extract the same features as week 3.
2. Write its `DatasetSpec` and load it with `standardise`.
3. Use a dataset with real within-person label change, and a contact reference to train its gate.
4. Rerun sections 2 to 8.

A model trained on MCD must not be deployed elsewhere unless it is approved on MCD and also shows skill on the new dataset.

In [ ]:
S = pd.DataFrame([{k: v[k] for k in ('target','model','skill','lo','hi','coverage','approved','reason')} for v in V.values()]).round(2)
S.to_csv(f'{OUT}/summary.csv', index=False); print(GR.to_string()); S

auc              0.969
coverage         0.811
before_median    0.732
before_mean      5.529
before_bad       0.161
after_median     0.586
after_mean       1.017
after_bad        0.029


,target,model,skill,lo,hi,coverage,approved,reason
0,stress,hgb,-1.86,-3.13,-0.43,0.81,False,signal below detection limit
1,temperature,median_ridge,-2.80,-4.42,-1.04,0.81,False,signal below detection limit
